In [2]:
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
validator_exits_size = pd.read_csv('../int/validator_exits_size.csv')
validator_exits_category = pd.read_csv('../int/validator_exits_category.csv')
validator_exits_pool = pd.read_csv('../int/validator_exits_pool.csv')
active_validators_size = pd.read_csv('../int/active_validators_size.csv')
active_validators_category = pd.read_csv('../int/active_validators_category.csv')
active_validators_pool = pd.read_csv('../int/active_validators_pool.csv')

active_validators_size = active_validators_size[active_validators_size['slot'] >= 6206400]
active_validators_category = active_validators_category[active_validators_category['slot'] >= 6206400]
active_validators_pool = active_validators_pool[active_validators_pool['slot'] >= 6206400]

In [3]:
rewards_size = pd.read_csv('../int/rewards_size.csv').drop(columns=['epoch'])
rewards_category = pd.read_csv('../int/rewards_category.csv').drop(columns=['epoch'])
rewards_pool = pd.read_csv('../int/rewards_pool.csv').drop(columns=['epoch'])
rewards_size = rewards_size[rewards_size['slot'].isin(validator_exits_size['slot'])]
rewards_category = rewards_category[rewards_category['slot'].isin(validator_exits_category['slot'])]
rewards_pool = rewards_pool[rewards_pool['slot'].isin(validator_exits_pool['slot'])]

rewards_size = rewards_size.drop(columns=('total'))
rewards_category = rewards_category.drop(columns=('total'))
rewards_pool = rewards_pool.drop(columns=('total'))

rewards_size['total'] = rewards_size.drop(columns=['slot']).mean(axis=1)
rewards_category['total'] = rewards_category.drop(columns=['slot']).mean(axis=1)
rewards_pool['total'] = rewards_pool.drop(columns=['slot']).mean(axis=1)

In [4]:
rewards_size

,1,100+,2-5,20-99,6-19,slot,total
0,3.228596,3.523359,3.328382,3.354958,3.360861,6840000.0,3.359231
1,3.191046,3.516477,3.327047,3.421507,3.397688,6847200.0,3.370753
2,3.165034,3.513273,3.310877,3.441939,3.375904,6854400.0,3.361406
3,3.160333,3.498146,3.318798,3.479943,3.481446,6861600.0,3.387733
4,3.126423,3.497919,3.337881,3.434020,3.352922,6868800.0,3.349833
...,...,...,...,...,...,...,...
294,2.672540,2.855028,2.707536,2.854167,2.794916,8956800.0,2.776838
295,2.826393,2.862574,2.734790,2.859195,2.788211,8964000.0,2.814233
296,2.690608,2.864407,2.592743,2.893811,2.807954,8971200.0,2.769905
297,2.696689,2.873085,2.662818,2.872214,2.802212,8978400.0,2.781404


In [5]:
validator_exits_size_set = validator_exits_size
validator_exits_size_set['slot'] = validator_exits_size_set['slot'] // 7200 * 7200

# Group by the day and get the last slot of each day and sum values for all columns
validator_exits_size_set = validator_exits_size_set.groupby('slot').agg(
    {
        'slot': 'last',
        '1': 'sum',
        '2-5': 'sum',
        '6-19': 'sum',
        '20-99': 'sum',
        '100+': 'sum',
        'total': 'sum'
    }
).reset_index(drop=True)

validator_exits_size_set

,slot,1,2-5,6-19,20-99,100+,total
0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,7200.0,0.0,0.0,0.0,0.0,0.0,0.0
2,14400.0,0.0,0.0,0.0,0.0,10.0,10.0
3,21600.0,0.0,0.0,1.0,0.0,0.0,1.0
4,28800.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
1244,8956800.0,2.0,10.0,7.0,8.0,437.0,464.0
1245,8964000.0,11.0,7.0,10.0,6.0,1447.0,1481.0
1246,8971200.0,99.0,11.0,19.0,96.0,1151.0,1376.0
1247,8978400.0,9.0,8.0,1.0,7.0,669.0,694.0


In [6]:
validator_exits_category_set = validator_exits_category
validator_exits_category_set['slot'] = validator_exits_category_set['slot'] // 7200 * 7200

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in validator_exits_category_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
validator_exits_category_set = validator_exits_category_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

validator_exits_category_set

,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total,slot
0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7200.0
2,0.0,0.0,10.0,0.0,0.0,0.0,10.0,14400.0
3,0.0,0.0,0.0,0.0,0.0,1.0,1.0,21600.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28800.0
...,...,...,...,...,...,...,...,...
1244,310.0,0.0,16.0,4.0,111.0,23.0,464.0,8956800.0
1245,300.0,361.0,150.0,0.0,568.0,102.0,1481.0,8964000.0
1246,341.0,50.0,188.0,2.0,20.0,775.0,1376.0,8971200.0
1247,94.0,0.0,182.0,0.0,34.0,384.0,694.0,8978400.0


In [7]:
validator_exits_pool_set = validator_exits_pool
validator_exits_pool_set['slot'] = validator_exits_pool_set['slot'] // 7200 * 7200

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in validator_exits_pool_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
validator_exits_pool_set = validator_exits_pool_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

validator_exits_pool_set

,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total,slot
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7200.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,10.0,14400.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,21600.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1244,0.0,0.0,47.0,0.0,6.0,10.0,0.0,100.0,0.0,294.0,7.0,464.0,8956800.0
1245,0.0,0.0,200.0,361.0,3.0,534.0,0.0,33.0,0.0,338.0,12.0,1481.0,8964000.0
1246,0.0,2.0,320.0,50.0,13.0,14.0,145.0,6.0,0.0,803.0,23.0,1376.0,8971200.0
1247,0.0,0.0,84.0,0.0,3.0,17.0,132.0,7.0,0.0,407.0,44.0,694.0,8978400.0


In [8]:
active_validators_size_set = active_validators_size[active_validators_size['slot'].isin(validator_exits_size_set['slot'])]
active_validators_size_set.reset_index(inplace=True)
active_validators_size_set = active_validators_size_set.drop(columns=['index'])
active_validators_size_set

,slot,1,100+,2-5,20-99,6-19,total
0,6206400.0,6648.0,508415.0,8434.0,26693.0,12655.0,562845.0
1,6213600.0,6622.0,507748.0,8434.0,26846.0,12671.0,562321.0
2,6220800.0,6623.0,507027.0,8447.0,26891.0,12667.0,561655.0
3,6228000.0,6629.0,506934.0,8452.0,26964.0,12676.0,561655.0
4,6235200.0,6633.0,506690.0,8453.0,27176.0,12703.0,561655.0
...,...,...,...,...,...,...,...
382,8956800.0,9865.0,917054.0,9915.0,44138.0,17998.0,998970.0
383,8964000.0,9894.0,918299.0,9914.0,44152.0,18033.0,1000292.0
384,8971200.0,9898.0,918498.0,9920.0,44277.0,18033.0,1000626.0
385,8978400.0,9824.0,918915.0,9926.0,44343.0,18042.0,1001050.0


In [9]:
active_validators_category_set = active_validators_category[active_validators_category['slot'].isin(validator_exits_category_set['slot'])]
active_validators_category_set.reset_index(inplace=True)
active_validators_category_set = active_validators_category_set.drop(columns=['index'])
active_validators_category_set

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,6206400.0,197661.0,8.0,203116.0,21557.0,26183.0,114320.0,562845.0
1,6213600.0,196665.0,8.0,203302.0,21564.0,26201.0,114581.0,562321.0
2,6220800.0,195247.0,8.0,203313.0,21564.0,26261.0,115262.0,561655.0
3,6228000.0,194313.0,8.0,203187.0,21575.0,26304.0,116268.0,561655.0
4,6235200.0,193376.0,8.0,203190.0,21618.0,26385.0,117078.0,561655.0
...,...,...,...,...,...,...,...,...
382,8956800.0,255557.0,75916.0,331402.0,16812.0,57077.0,262206.0,998970.0
383,8964000.0,255875.0,76685.0,331441.0,16819.0,56993.0,262479.0,1000292.0
384,8971200.0,256042.0,77041.0,331314.0,16821.0,56440.0,262968.0,1000626.0
385,8978400.0,255922.0,77313.0,331173.0,16827.0,56445.0,263370.0,1001050.0


In [10]:
active_validators_pool_set = active_validators_pool[active_validators_pool['slot'].isin(validator_exits_pool_set['slot'])]
active_validators_pool_set.reset_index(inplace=True)
active_validators_pool_set = active_validators_pool_set.drop(columns=['index'])
active_validators_pool_set

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,6206400.0,35351.0,13677.0,84118.0,0.0,38985.0,2332.0,177413.0,0.0,5322.0,191339.0,14308.0,562845.0
1,6213600.0,35351.0,13677.0,84214.0,0.0,37977.0,2350.0,177554.0,0.0,5335.0,191520.0,14343.0,562321.0
2,6220800.0,35359.0,13677.0,84283.0,0.0,36305.0,2409.0,177583.0,0.0,5435.0,192261.0,14343.0,561655.0
3,6228000.0,35373.0,13677.0,84680.0,0.0,34917.0,2451.0,177829.0,0.0,5435.0,192953.0,14340.0,561655.0
4,6235200.0,35373.0,13677.0,84857.0,0.0,33416.0,2499.0,178014.0,0.0,5661.0,193811.0,14347.0,561655.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,8956800.0,35167.0,18316.0,138553.0,33886.0,23668.0,15216.0,290491.0,15115.0,10877.0,392976.0,24705.0,998970.0
383,8964000.0,35167.0,18665.0,138721.0,33886.0,23693.0,15226.0,290491.0,15020.0,10877.0,393830.0,24716.0,1000292.0
384,8971200.0,35167.0,19008.0,138579.0,33525.0,23717.0,14697.0,290491.0,14987.0,10877.0,394865.0,24713.0,1000626.0
385,8978400.0,35167.0,19013.0,138400.0,33475.0,23732.0,14707.0,290346.0,14981.0,10877.0,395661.0,24691.0,1001050.0


In [11]:
validator_exits_size_set.set_index('slot', inplace=True)
active_validators_size_set.set_index('slot', inplace=True)
validator_exit_percentage_size = validator_exits_size_set.divide(active_validators_size_set, fill_value=0) * 100
validator_exit_percentage_size.reset_index(inplace=True)
validator_exit_percentage_size

,slot,1,100+,2-5,20-99,6-19,total
0,0.0,NaN,NaN,NaN,inf,NaN,inf
1,7200.0,NaN,NaN,NaN,NaN,NaN,NaN
2,14400.0,NaN,inf,NaN,NaN,NaN,inf
3,21600.0,NaN,NaN,NaN,NaN,inf,inf
4,28800.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1244,8956800.0,0.020274,0.047653,0.100857,0.018125,0.038893,0.046448
1245,8964000.0,0.111178,0.157574,0.070607,0.013589,0.055454,0.148057
1246,8971200.0,1.000202,0.125313,0.110887,0.216817,0.105362,0.137514
1247,8978400.0,0.091612,0.072803,0.080596,0.015786,0.005543,0.069327


In [12]:
validator_exits_category_set.set_index('slot', inplace=True)
active_validators_category_set.set_index('slot', inplace=True)
validator_exit_percentage_category = validator_exits_category_set.divide(active_validators_category_set, fill_value=0) * 100
validator_exit_percentage_category.reset_index(inplace=True)
validator_exit_percentage_category

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,0.0,NaN,NaN,NaN,NaN,NaN,inf,inf
1,7200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14400.0,NaN,NaN,inf,NaN,NaN,NaN,inf
3,21600.0,NaN,NaN,NaN,NaN,NaN,inf,inf
4,28800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1244,8956800.0,0.121304,0.000000,0.004828,0.023793,0.194474,0.008772,0.046448
1245,8964000.0,0.117245,0.470757,0.045257,0.000000,0.996614,0.038860,0.148057
1246,8971200.0,0.133181,0.064901,0.056744,0.011890,0.035436,0.294713,0.137514
1247,8978400.0,0.036730,0.000000,0.054956,0.000000,0.060236,0.145802,0.069327


In [13]:
validator_exits_pool_set.set_index('slot', inplace=True)
active_validators_pool_set.set_index('slot', inplace=True)
validator_exit_percentage_pool = validator_exits_pool_set.divide(active_validators_pool_set, fill_value=0) * 100
validator_exit_percentage_pool.reset_index(inplace=True)
validator_exit_percentage_pool

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,inf,NaN,inf
1,7200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14400.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,inf,NaN,inf
3,21600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,inf,NaN,inf
4,28800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1244,8956800.0,0.0,0.000000,0.033922,0.000000,0.025351,0.065720,0.000000,0.661594,0.0,0.074814,0.028334,0.046448
1245,8964000.0,0.0,0.000000,0.144174,1.065337,0.012662,3.507159,0.000000,0.219707,0.0,0.085824,0.048552,0.148057
1246,8971200.0,0.0,0.010522,0.230915,0.149142,0.054813,0.095258,0.049915,0.040035,0.0,0.203361,0.093068,0.137514
1247,8978400.0,0.0,0.000000,0.060694,0.000000,0.012641,0.115591,0.045463,0.046726,0.0,0.102866,0.178203,0.069327


In [14]:
columns = ['1', '100+', '2-5', '20-99', '6-19', 'total']
for col in columns:
    print(validator_exit_percentage_size[col].mean())

inf
inf
inf
inf
inf
inf


In [15]:
# Calculate the percent change of APY
rewards_size_pct_change = rewards_size.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(validator_exit_percentage_size, rewards_size_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total_x'] / price_elasticity_size['total_y']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_size[elasticity_col_name] = price_elasticity_size[f'{col}_x'] / price_elasticity_size[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_size[elasticity_col_name] = price_elasticity_size[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)


Elasticity Analysis Results:
total: Mean Elasticity = 14.43075062320502, t(Mean) = -, SD = 218.44849968292257, N = 298, p-value = -
1: Mean Elasticity = 5.008218557020306, t(Mean) = -0.7124824211264079, SD = 66.33331544183793, N = 298, p-value = 0.47663901682681165
2-5: Mean Elasticity = 12.453285612849246, t(Mean) = -0.10197084272102445, SD = 253.6696970896345, N = 298, p-value = 0.9188149881867056
6-19: Mean Elasticity = -8.329419222154803, t(Mean) = -1.5082849823355549, SD = 141.9089778855339, N = 298, p-value = 0.13210107933168683
20-99: Mean Elasticity = 106.5237808651607, t(Mean) = 0.8716324402910723, SD = 1810.7730166046922, N = 298, p-value = 0.3840930773425588
100+: Mean Elasticity = 123.29347029523706, t(Mean) = 0.687766479744908, SD = 2723.6666104041087, N = 298, p-value = 0.49212979980381066
          slot       1_x    100+_x     2-5_x   20-99_x    6-19_x   total_x  \
0    6840000.0  0.075053  0.008389  0.034742  0.066540  0.000000  0.011904   
1    6847200.0  0.045045  0.1

In [16]:
# Calculate the percent change of APY
rewards_category_pct_change = rewards_category.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(validator_exit_percentage_category, rewards_category_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total_x'] / price_elasticity_category['total_y']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column
columns = [col for col in validator_exit_percentage_category.columns if col != 'slot']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_category[elasticity_col_name] = price_elasticity_category[f'{col}_x'] / price_elasticity_category[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_category[elasticity_col_name] = price_elasticity_category[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = 2.273101773817155, t(Mean) = 0.0, SD = 187.9269900206388, N = 298, p-value = 1.0
CEX: Mean Elasticity = -12.975285173824219, t(Mean) = -0.44500357403166785, SD = 560.8724570652432, N = 298, p-value = 0.6565825082121977
Liquid Restaking: Mean Elasticity = 0.4965031144039729, t(Mean) = -0.1616977294382291, SD = 25.638053883194875, N = 298, p-value = 0.8716498941193855
Liquid Staking: Mean Elasticity = -5.198158650945178, t(Mean) = -0.4680623567162954, SD = 201.52048636754054, N = 298, p-value = 0.639912523052369
Solo Stakers: Mean Elasticity = -8.384818643688034, t(Mean) = -0.8093401155475408, SD = 127.90878981088824, N = 298, p-value = 0.41868719288991174
Staking Pools: Mean Elasticity = 0.5032720441876138, t(Mean) = -0.11260280205556777, SD = 195.70617872870955, N = 298, p-value = 0.9103835684729003
Unidentified: Mean Elasticity = 20.664773800750766, t(Mean) = 0.7831472059201078, SD = 359.21340512653774, N = 298, p-value = 0.4339542

In [17]:
# Calculate the percent change of APY
rewards_pool_pct_change = rewards_pool.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_pool = pd.merge(validator_exit_percentage_pool, rewards_pool_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['total_x'] / price_elasticity_pool['total_y']
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_pool[elasticity_col_name] = price_elasticity_pool[f'{col}_x'] / price_elasticity_pool[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_pool[elasticity_col_name] = price_elasticity_pool[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_pool[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = -39.400195953688964, t(Mean) = -, SD = 958.4784877157546, N = 298, p-value = -
Lido: Mean Elasticity = -6.942096112665031, t(Mean) = 0.5747216024306693, SD = 178.3498411941335, N = 298, p-value = 0.565886539090892
Coinbase: Mean Elasticity = 12.679964443269178, t(Mean) = 0.8313237299988899, SD = 500.87291759711036, N = 298, p-value = 0.4062336731078655
Binance: Mean Elasticity = -15.636882670020224, t(Mean) = 0.4203646038229731, SD = 183.3798528171777, N = 298, p-value = 0.6745023939120327
Rocketpool: Mean Elasticity = -6.124754111298906, t(Mean) = 0.5949083622780463, SD = 116.77484797652818, N = 298, p-value = 0.5523446430778505
Kraken: Mean Elasticity = 21.006126248229883, t(Mean) = 0.9895990841274606, SD = 437.8077247067437, N = 298, p-value = 0.322945680840109
OKX: Mean Elasticity = -9.533928954792678, t(Mean) = 0.5334462208389597, SD = 124.20346892510806, N = 298, p-value = 0.5941107490475512
Bitcoin Suisse: Mean Elasticity = 9

/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_24021/1512971578.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rewards_pool_pct_change = rewards_pool.set_index('slot').pct_change().reset_index()
